# 주파수 분석과 활동 분류 실습

In [3]:
# library import
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
import os 

# Helper Function

In [10]:
# 합성 신호 생성 함수
def generate_example_signal(dt : float = 0.1, end: float = 200.0):
    """ 
    # 09 주파수 분석 - 합성 전현파 생성
    # 두 개의 주시 성분 : 16초 주기 + 5초 주기
    """
    t = np.arange(0, end + dt, dt)
    x = np.cos(2 * np.pi * t / 16) + 0.75 * np.cos(2 * np.pi * t / 5)
    return t, x


#스펙트럼 계산 함수
def spectrum_like_r(x, fs: float, span : int = None):
    freq, pxx = signal.periodogram(x, fs = fs, detrend = False, scaling = 'density')
    # 양의 주파수만 선택
    pos = freq > 0
    freq, spec = freq[pos], 2 * pxx[pos]
    if span is not None and span >1:
        spec = pd.Series(spec).rolling(span, center = True).mean().to_numpy()
    return freq, spec


# 양의 주파수 선택 및 보정
def spectrum_like_r(x, fs, span = None):
    freq,pxx = signal.periodogram(x, fs=fs, return_onesided= False)
    positive = freq > 0
    freq, spec = freq[positive], 2 * pxx[positive]
    if span is not None: Spec = pd.Series(spec).rolling(span, center = True).mean()
    return freq, spec 

#트정 추출 함수 
def top_n_freq(freq, spec, n: int = 2):
    # 스펙트럼 크기(spec) 기준으로 내림차순 정렬된 인덱스
    idx = np.argsort(spec)[::-1][:n]
    
    #상위 n개의 주파수 와 스펙트럼 값을 DataFrame으로 반환
    return pd.DataFrame({'Frequency': freq[idx], 'Spectrum': spec[idx]}
                        ).sort_values('spectrum', ascending = False).reset_index(drop = True)
    
    
#시각화 함수 
data_path = os.getcwd()
output_dir = os.path.join(data_path, 'output')
_fig_counter = 0

def _savefig(name: str):
    """그래프를 output 디렉토리에 저장하는 함수"""
    _fig_counter[0] += 1
    path = output_dir/f"{_fig_counter[0]:02d}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi = 120, bbox_inches = 'tight')
    plt.close()
    print(f" -> 저장 완료 :{path.name}")
#시간-신호 그래프 생성
def plot_signal(t,x,title = 'Signal'):
    plt.figure(figsize = (12, 4))
    plt.plot(t,x)
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Amplitude(x)')
    plt.grid()
    plt.show()
    _savefig("signal")
    
#주파수-스펙트럼 그래프 생성
def plot_spectrum(freq, spec, title = 'Spectrum'):
    plt.figure(figsize = (12, 4))
    plt.plot(freq, spec)
    plt.title(title)
    plt.xlabel('Frequency')
    plt.ylabel('Spectrum(x)')
    plt.grid()
    plt.show()
    _savefig("spectrum")

# HAR_데이터

In [16]:
#HAR_total 파일 로드
Har_data_path = os.path.join(data_path , 'Data', 'A_DeviceMotion_data')
df = pd.read_pickle(Har_data_path + '/HAR_total.pkl')
display(df.head())

print(df.shape)
print(df.columns)

,Unnamed: 0,attitude.roll,attitude.pitch,attitude.yaw,gravity.x,gravity.y,gravity.z,rotationRate.x,rotationRate.y,rotationRate.z,userAcceleration.x,userAcceleration.y,userAcceleration.z,exp_no,id,activity,maguserAcceleration,magrotationRate
0,0,-2.116381,-1.077507,-2.261502,-0.404768,0.880780,0.245713,-1.264215,-1.027909,-0.947909,0.282683,-0.254346,-0.407670,11,12,dws,0.557491,1.885038
1,1,-2.148154,-1.049759,-2.284278,-0.417081,0.867303,0.271686,-1.162024,-0.269118,-0.848823,0.256712,0.079154,-0.560291,11,12,dws,0.621363,1.463976
2,2,-2.153824,-1.026749,-2.297008,-0.432082,0.855621,0.284961,-0.665042,0.520170,-0.726722,0.253600,0.346680,-0.463275,11,12,dws,0.631762,1.113994
3,3,-2.142509,-1.012749,-2.290595,-0.445311,0.848291,0.286507,-0.079809,0.055322,-0.604534,0.411818,0.459372,-0.510293,11,12,dws,0.800635,0.612284
4,4,-2.130486,-1.007262,-2.274149,-0.452661,0.845372,0.283600,0.456097,-0.186877,-0.441315,0.311594,0.477305,-0.925049,11,12,dws,1.086566,0.661594


(1412865, 18)
Index(['Unnamed: 0', 'attitude.roll', 'attitude.pitch', 'attitude.yaw',
       'gravity.x', 'gravity.y', 'gravity.z', 'rotationRate.x',
       'rotationRate.y', 'rotationRate.z', 'userAcceleration.x',
       'userAcceleration.y', 'userAcceleration.z', 'exp_no', 'id', 'activity',
       'maguserAcceleration', 'magrotationRate'],
      dtype='object')


In [17]:
#필토링 함수
def get_signal_by_group(df, column, activity=None, sub_id=None):
    """
    특정 activity와 subject(id)에 해당하는 시계열 데이터를 추출하는 함수
    """

    subset = df

    # activity 값이 주어지고, 'activity' 컬럼이 존재할 경우 해당 조건으로 필터링
    if activity is not None and "activity" in df.columns:
        subset = subset[subset["activity"] == activity]

    # sub_id 값이 주어지고, 'id' 컬럼이 존재할 경우 해당 조건으로 필터링
    if sub_id is not None and "id" in df.columns:
        subset = subset[subset["id"] == sub_id]

    # 선택한 column의 결측값 제거 후 numpy 배열 형태로 반환
    return subset[column].dropna().to_numpy()

def apply_highpass_filter(x, cutoff=0.05, order=1):
    """
    Butterworth 고역통과 필터를 적용하는 함수
    """

    # Butterworth 필터 계수(b, a) 생성
    b, a = signal.butter(order, cutoff, btype="highpass")

    # 입력 신호 x에 필터 적용 (선형 필터링)
    filtered = signal.lfilter(b, a, x)

    # 필터링된 신호와 필터 계수 반환
    return filtered, (b, a)

def extract_top_n_freq_features(x, fs: float, n: int = 5, span: int = 10):
    """
    스펙트럼에서 상위 N개의 주파수 특징을 추출하는 함수
    """

    # 주파수(freq)와 스펙트럼 크기(spec) 계산
    freq, spec = spectrum_like_(x, fs=fs, span=span)

    # 스펙트럼 크기 기준으로 내림차순 정렬 후 상위 n개 인덱스 선택
    idx = np.argsort(spec)[::-1][:n]

    # 선택된 인덱스를 주파수 기준으로 다시 오름차순 정렬
    idx = idx[np.argsort(freq[idx])]

    features = {}

    # 각 상위 주파수와 해당 스펙트럼 값을 feature로 저장
    for i, j in enumerate(idx, start=1):
        features[f"f{i}"] = float(freq[j])        # 주파수 값
        features[f"spectrum{i}"] = float(spec[j]) # 해당 스펙트럼 크기

    return features


def demo_batch_feature_extraction_pkl(df, column="magrotationRate", fs=50.0):
    """
    그룹별로 상위 5개 주파수 특징을 추출하는 배치 함수 (pkl 데이터용)
    """

    # 그룹 기준 컬럼 설정 (존재하는 컬럼만 사용)
    group_keys = [c for c in ["activity", "id"] if c in df.columns]

    rows = []

    # 그룹 단위로 데이터 순회
    for i, (keys, grp) in enumerate(df.groupby(group_keys), start=1):

        # 선택한 컬럼의 시계열 데이터 추출 (결측값 제거)
        x = grp[column].dropna().to_numpy()

        # 데이터 길이가 너무 짧으면 건너뜀
        if len(x) < 20:
            continue

        # 상위 주파수 특징 추출
        features = extract_top_n_freq_features(x, fs=fs, n=5, span=10)

        # 그룹 키를 딕셔너리 형태로 변환
        row = dict(zip(group_keys, keys if isinstance(keys, tuple) else [keys]))

        # 특징값 병합
        row.update(features)

        # 결과 리스트에 추가
        rows.append(row)

    # 최종 결과를 DataFrame으로 변환
    return pd.DataFrame(rows)